# Pack Scientific Units into Workers

Scientific units and worker packs solve different problems. `artifacts_per_unit` changes the finite scientific partition. `units_per_worker` groups compatible units into one worker that executes them sequentially, preserving each unit's independent result and cleanup boundary.

Compare one unit per worker with three units per worker. Both runs must produce the same transformed contents. The elapsed times are observations, not a portable speedup assertion.

In [ ]:
from __future__ import annotations

import time

from artisan.inspection import inspect_evidence
from artisan.operations.examples import DataGenerator, DataTransformer
from artisan.orchestration import PipelineManager, StepDisposition
from artisan.storage import ArtifactStore
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [ ]:
from functools import partial
from tempfile import TemporaryDirectory

# Keep this tutorial's stores and source files in one owned temporary workspace.
workspace = TemporaryDirectory(prefix="artisan-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)
env = tutorial_setup("worker_packs")

In [ ]:
with PipelineManager.create(
    name="worker_packs",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    skip_cache=True,
) as pipeline:
    pipeline.run(DataGenerator, name="generate", params={"count": 6, "seed": 42})
    steps = []
    for pack_size in (1, 3):
        started = time.perf_counter()
        step = pipeline.run(
            DataTransformer,
            name=f"pack_{pack_size}",
            inputs={"dataset": pipeline.output("generate", "datasets")},
            params={"scale_factor": 0.5, "variants": 1, "seed": 100},
            batch_strategy={
                "artifacts_per_unit": 1,
                "units_per_worker": pack_size,
                "max_workers": 2,
            },
        )
        steps.append(step)
        print(f"{pack_size} unit(s) per worker: {time.perf_counter() - started:.2f}s")
    summary = pipeline.finalize()
assert summary["overall_success"]
assert all(step.total_count == 6 and step.succeeded_count == 6 for step in steps)
assert all(step.disposition is StepDisposition.EXECUTED for step in steps)

In [ ]:
refs = inspect_evidence(
    "query_artifacts",
    {
        "pipeline_run_id": pipeline.config.pipeline_run_id,
        "artifact_type": "csv",
        "limit": 100,
    },
    delta_root=env.delta_root,
)
store = ArtifactStore(env.delta_root)
contents = []
for step in steps:
    ids = [
        item.artifact_id
        for item in refs.items
        if item.current_step_number == step.step_number
    ]
    artifacts = store.get_artifacts_by_type(ids, "csv")
    assert len(artifacts) == 6
    contents.append({artifact.content for artifact in artifacts.values()})
assert contents[0] == contents[1]
inspect_pipeline(env.delta_root, pipeline_run_id=pipeline.config.pipeline_run_id)

Packing does not merge cache identities or publish a combined scientific result. If one unit fails, the runner retains independent outcomes and ownership evidence for the other units. Larger packs reduce startup overhead but hold one worker allocation for longer.